In [29]:
import pandas as pd

df = pd.read_csv('/home/ji/NBA_Project/data/data_04_03_2025.csv')

df.head()

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,22024,1610612737,ATL,Atlanta Hawks,22400878,2025-03-03,ATL @ MEM,W,240,132,...,0.529,14.0,18.0,32.0,35,12.0,7,12,24,2.0
1,22024,1610612737,ATL,Atlanta Hawks,22400851,2025-02-28,ATL vs. OKC,L,240,119,...,0.783,10.0,36.0,46.0,27,8.0,5,19,14,-16.0
2,22024,1610612737,ATL,Atlanta Hawks,22400841,2025-02-26,ATL @ MIA,L,240,109,...,0.789,9.0,24.0,33.0,29,6.0,3,15,15,-22.0
3,22024,1610612737,ATL,Atlanta Hawks,22400825,2025-02-24,ATL vs. MIA,W,240,98,...,0.750,13.0,37.0,50.0,29,13.0,2,16,22,12.0
4,22024,1610612737,ATL,Atlanta Hawks,22400814,2025-02-23,ATL vs. DET,L,240,143,...,0.722,10.0,30.0,40.0,30,8.0,6,13,21,-5.0


In [30]:
df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
df = df[df['GAME_DATE'] >='2014-08-01']

df = df.dropna()

df['WL'] = df['WL'].str.strip()

df['WL'] = df['WL'].map({'W':1,'L':0})

df['HOME_AWAY'] = df['MATCHUP'].apply(lambda x: 'A' if '@' in x else 'H')

In [31]:
df = df[df['HOME_AWAY']=='A']

In [32]:
df = df.sort_values(['TEAM_ID','GAME_DATE'])

In [33]:
 # List of stat columns for which to compute last 5 games average
stat_columns = [
        "PTS", "FGM", "FGA", "FG_PCT", "FG3M", "FG3A", "FG3_PCT",
        "FTM", "FTA", "FT_PCT", "OREB", "DREB", "REB", "AST",
        "STL", "BLK", "TOV", "PF", "PLUS_MINUS"
    ]

In [34]:
column_list = df.columns.tolist()

print(column_list)

['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS', 'HOME_AWAY']


In [35]:
# For each column, compute the rolling average of the last 5 games (excluding the current game) 
for col_name in stat_columns:
    new_col = col_name + "_LAST5"
    df[new_col] = (df.groupby("TEAM_ID")[col_name]
                         .apply(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
                         .reset_index(level=0, drop=True))

df = df.dropna()

In [36]:
stat_columns_away = [f"{col}_LAST5" for col in stat_columns]
columns = stat_columns_away +  ['WL']
df = df[columns]

df.head()

,PTS_LAST5,FGM_LAST5,FGA_LAST5,FG_PCT_LAST5,FG3M_LAST5,FG3A_LAST5,FG3_PCT_LAST5,FTM_LAST5,FTA_LAST5,FT_PCT_LAST5,OREB_LAST5,DREB_LAST5,REB_LAST5,AST_LAST5,STL_LAST5,BLK_LAST5,TOV_LAST5,PF_LAST5,PLUS_MINUS_LAST5,WL
1029,88.000000,32.0,75.000000,0.427000,8.000000,25.000000,0.320000,16.000000,22.000000,0.72700,9.0,33.000000,42.000000,22.000000,4.000000,7.000000,15.00,21.000000,-5.0,1
1028,98.500000,35.5,77.500000,0.457500,10.500000,29.500000,0.351000,17.000000,23.000000,0.73850,9.0,34.500000,43.500000,24.500000,4.500000,7.000000,18.00,21.500000,0.5,0
1025,93.666667,34.0,78.333333,0.434333,9.333333,28.666667,0.320333,16.333333,21.333333,0.77000,9.0,35.333333,44.333333,23.333333,5.333333,6.666667,17.00,24.333333,0.0,1
1024,99.500000,35.5,78.000000,0.455500,10.000000,27.750000,0.360250,18.500000,24.500000,0.76125,8.5,33.250000,41.750000,24.750000,5.500000,5.500000,16.25,23.750000,2.5,0
1022,100.000000,36.4,78.400000,0.464400,10.600000,26.600000,0.406400,16.600000,23.000000,0.71480,8.8,33.000000,41.800000,25.000000,5.600000,6.000000,16.40,23.800000,0.6,0


In [37]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from datetime import datetime
import joblib

# Get today's date in dd_mm_yyyy format
today_str = datetime.today().strftime("%d_%m_%Y")

# Construct the filename
#filename = "/home/ji/NBA_Project/data/"+f"data_cleaned_{today_str}.csv"

#df = pd.read_csv(filename)


X = df[stat_columns_away]
y = df['WL']


# Split data into training and testing sets (70/30 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Create and train the logistic regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Generate predictions and predicted probabilities on the test set
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Evaluate the model using AUC (Area Under the ROC Curve)
auc = roc_auc_score(y_test, y_prob)
print(f"Model AUC: {auc}")

# Optionally, display a few prediction results
results = X_test.copy()
results['WIN'] = y_test
results['Prediction'] = y_pred
results['Probability'] = y_prob
print(results.head())


# Save the trained model to a file
model_filename = "/home/ji/NBA_Project/src/models/model_logistic_" + today_str + ".pkl"
joblib.dump(model, model_filename)
print(f"Model saved to: {model_filename}")

Model AUC: 0.5804259504232104
       PTS_LAST5  FGM_LAST5  FGA_LAST5  FG_PCT_LAST5  FG3M_LAST5  FG3A_LAST5  \
17533       95.0       34.6       77.8        0.4494        11.8        33.8   
40560      107.6       38.0       84.6        0.4504        13.0        34.6   
66237       86.8       32.6       82.4        0.3974         6.2        22.6   
21845      103.0       39.4       88.4        0.4462        12.4        34.0   
65431      124.0       46.0       91.4        0.5050        12.0        32.2   

       FG3_PCT_LAST5  FTM_LAST5  FTA_LAST5  FT_PCT_LAST5  ...  REB_LAST5  \
17533         0.3542       14.0       19.4        0.7112  ...       37.4   
40560         0.3816       18.6       22.2        0.8380  ...       42.2   
66237         0.2782       15.4       20.6        0.7508  ...       46.4   
21845         0.3474       11.8       16.0        0.7594  ...       42.4   
65431         0.3700       20.0       25.4        0.7810  ...       45.0   

       AST_LAST5  STL_LAST5  BLK